# Which function is missing? — reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alestainer/statistics-intuitions/blob/main/notebooks/04-support-boundary.ipynb)

Reproduces the 20 fixed support-boundary questions, exact prompts and saved model decisions used in the article. No paid request runs automatically.


In [ ]:
from pathlib import Path
import json, urllib.request

RAW = "https://raw.githubusercontent.com/Alestainer/statistics-intuitions/main/"

def load_json(relative_path):
    candidates = [Path("../") / relative_path, Path(relative_path)]
    for path in candidates:
        if path.exists():
            return json.loads(path.read_text())
    with urllib.request.urlopen(RAW + relative_path) as response:
        return json.load(response)

benchmark = load_json("data/04-support-boundary/benchmark.json")
results = load_json("data/04-support-boundary/results.json")
print(f"{len(benchmark['episodes'])} questions; {len(results['models'])} completed model runs")


## Inspect the exact images and options

In [ ]:
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

def read_bytes(relative_path):
    for path in (Path("../") / relative_path, Path(relative_path)):
        if path.exists(): return path.read_bytes()
    with urllib.request.urlopen(RAW + relative_path) as response: return response.read()

def episode_image(episode):
    name = Path(episode["image"]).name
    return Image.open(BytesIO(read_bytes("data/04-support-boundary/stimuli/" + name))).convert("RGB")

episode = benchmark["episodes"][0]
plt.figure(figsize=(10, 6)); plt.imshow(episode_image(episode)); plt.axis("off"); plt.show()
print(episode["userPrompt"])
print("Expected:", episode["answer"])


## Verify deterministic generation

In [ ]:
import math

RECT = {"xMin": -20., "xMax": 20., "yMin": -20., "yMax": 20.}

def hash_string(value):
    result = 2166136261
    for char in value:
        result ^= ord(char); result = (result * 16777619) & 0xffffffff
    return result

class Mulberry32:
    def __init__(self, seed): self.state = seed & 0xffffffff
    def __call__(self):
        self.state = (self.state + 0x6D2B79F5) & 0xffffffff
        value = self.state
        value = ((value ^ (value >> 15)) * (value | 1)) & 0xffffffff
        value ^= (value + (((value ^ (value >> 7)) * (value | 61)) & 0xffffffff)) & 0xffffffff
        return ((value ^ (value >> 14)) & 0xffffffff) / 4294967296

FUNCTIONS = {
    "line": lambda x: x, "negative-line": lambda x: -3*x,
    "parabola": lambda x: x*x, "sine": lambda x: 10*math.sin(x),
    "exp-sine": lambda x: math.exp(x/7)*math.sin(x),
    "log": lambda x: math.log(x) if x > 0 else math.nan,
    "exponential": lambda x: 2**x,
}

def is_above(function_id, x, y):
    return x < math.exp(y) if function_id == "log" else y > FUNCTIONS[function_id](x)

def regenerate_points(episode):
    rng = Mulberry32(hash_string(f'{episode["seed"]}:{episode["pointCount"]}'))
    function_ids = [f["id"] for f in benchmark["generation"]["functions"]]
    hidden = function_ids[math.floor(rng() * len(function_ids))]
    side = "above" if rng() < .5 else "below"
    points = []
    while len(points) < episode["pointCount"]:
        x = RECT["xMin"] + rng() * 40; y = RECT["yMin"] + rng() * 40
        above = is_above(hidden, x, y)
        if (side == "above" and above) or (side == "below" and not above): points.append((x, y))
    return hidden, side, points

hidden, side, points = regenerate_points(episode)
assert hidden == episode["hiddenFunctionId"] and side == episode["side"] and len(points) == 200
print("Regenerated:", hidden, side, len(points), "points")


## Exact model prompt

In [ ]:
print(benchmark["systemPrompt"])
print("\n--- example user message ---\n")
print(episode["userPrompt"])


## Recompute the published table

In [ ]:
import pandas as pd

rows = []
for run in results["models"]:
    episodes = run["episodes"]
    rows.append({
        "model": run["model"],
        "correct": f'{sum(e["correct"] for e in episodes)}/20',
        "invalid responses": sum(not e["strictlyFormatted"] for e in episodes),
        "cost (USD)": round(sum(e["costUsd"] for e in episodes), 4),
    })
pd.DataFrame(rows).sort_values("correct", ascending=False).reset_index(drop=True)


## Full opt-in API replication

The cells below run the exact published tasks across all published models (20 episodes × 6 models), save every request and raw response, record provider, tokens, latency and cost, and resume from completed calls.

Paid calls are off by default. Add `OPENROUTER_API_KEY` to Colab Secrets, set `RUN_PAID_EVAL = True`, and choose an aggregate budget. The historical run cost about $0.44; current prices and routing can differ. Each request reserves a conservative worst-case amount before it is sent, so the configured aggregate budget cannot be exceeded.


In [ ]:
import base64, hashlib, re

JOBS = [{**episode, "id": episode["id"], "image_bytes": read_bytes("data/04-support-boundary/stimuli/" + Path(episode["image"]).name)} for episode in benchmark["episodes"]]
BENCHMARK_ID = benchmark["benchmarkId"]
RUN_PAID_EVAL = False
BUDGET_USD = 5.0
RERUN_DIR = Path("support-boundary-rerun")
MODEL_SPECS = {
    "openai/gpt-5.6-sol": {"input": 2.0, "output": 10.0},
    "google/gemini-3.7-flash": {"input": 0.75, "output": 3.75},
    "anthropic/claude-opus-5": {"input": 5.0, "output": 25.0},
    "anthropic/claude-sonnet-5": {"input": 2.0, "output": 10.0},
    "qwen/qwen3.8-max": {"input": 2.0, "output": 6.0},
    "moonshotai/kimi-k3": {"input": 3.0, "output": 15.0},
    "z-ai/glm-5.3-flash": {"input": 0.075, "output": 0.25},
}
MODELS_TO_RUN = [run["model"] for run in results["models"]]
RUN_SETTINGS = {"max_tokens": 8192, "reasoning": {"effort": "low"}, "temperature": 0}

def parse_command(text):
    normalized = (text or "").strip().upper()
    valid = set(benchmark["validCommands"])
    return (normalized if normalized in valid else None), normalized in valid

def build_payload(model, job):
    image = base64.b64encode(job["image_bytes"]).decode()
    messages = [
        {"role": "system", "content": benchmark["systemPrompt"]},
        {"role": "user", "content": [
            {"type": "text", "text": job["userPrompt"]},
            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + image}},
        ]},
    ]
    return openrouter_payload(model, messages, **RUN_SETTINGS)

def job_metadata(job):
    return {"episode_id": job["id"], "image_sha256": hashlib.sha256(job["image_bytes"]).hexdigest()}


In [ ]:
import hashlib, os, time, urllib.error
from datetime import datetime, timezone

OPENROUTER_ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"
MAX_NOTEBOOK_BUDGET_USD = 25.0

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(encoded.encode()).hexdigest()

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")

def append_jsonl(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as handle:
        handle.write(json.dumps(value, ensure_ascii=False) + "\n")

def get_api_key():
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            key = None
    if not key:
        import getpass
        key = getpass.getpass("OpenRouter API key: ")
    return key

def openrouter_payload(model, messages, *, max_tokens, reasoning, temperature=None):
    spec = MODEL_SPECS[model]
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "reasoning": reasoning,
        "usage": {"include": True},
        "provider": {
            "sort": "price", "allow_fallbacks": False, "require_parameters": False,
            "max_price": {"prompt": spec["input"], "completion": spec["output"]},
        },
    }
    if temperature is not None:
        payload["temperature"] = temperature
    return payload

def reserve_cost(payload):
    spec = MODEL_SPECS[payload["model"]]
    # UTF-8 bytes safely overestimate prompt tokens, including encoded images.
    prompt_ceiling = len(json.dumps(payload["messages"], ensure_ascii=False).encode()) + 256
    return prompt_ceiling * spec["input"] / 1_000_000 + payload["max_tokens"] * spec["output"] / 1_000_000

def send_openrouter(payload, api_key, timeout=240):
    request = urllib.request.Request(
        OPENROUTER_ENDPOINT,
        data=json.dumps(payload).encode(),
        method="POST",
        headers={
            "Authorization": f"Bearer {api_key}", "Content-Type": "application/json",
            "HTTP-Referer": "https://alestainer.com", "X-OpenRouter-Title": BENCHMARK_ID,
        },
    )
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status, body = response.status, response.read().decode()
    except urllib.error.HTTPError as error:
        status, body = error.code, error.read().decode(errors="replace")
    try:
        parsed = json.loads(body)
    except json.JSONDecodeError:
        parsed = {"unparsed_body": body}
    return status, parsed, time.perf_counter() - started

def content_of(response):
    try:
        content = response["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        return None
    return content if isinstance(content, str) else None

def usage_of(response):
    usage = response.get("usage") if isinstance(response.get("usage"), dict) else {}
    details = usage.get("completion_tokens_details")
    details = details if isinstance(details, dict) else {}
    cost = usage.get("cost")
    reasoning = details.get("reasoning_tokens")
    return {
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "reasoning_tokens": reasoning if isinstance(reasoning, (int, float)) else None,
        "cost_usd": cost if isinstance(cost, (int, float)) else None,
        "raw_usage": usage,
    }

def result_files():
    return sorted(RERUN_DIR.glob("models/*/calls/*/result.json"))

def load_existing():
    existing = {}
    for path in result_files():
        row = json.loads(path.read_text())
        if row.get("suite_hash") != SUITE_HASH:
            raise RuntimeError(f"Suite mismatch in {path}")
        existing[(row["model"], row["call_id"])] = row
    return existing

def prepare_run():
    if BUDGET_USD <= 0 or BUDGET_USD > MAX_NOTEBOOK_BUDGET_USD:
        raise ValueError(f"BUDGET_USD must be above $0 and at most ${MAX_NOTEBOOK_BUDGET_USD}")
    RERUN_DIR.mkdir(parents=True, exist_ok=True)
    config_path = RERUN_DIR / "config.json"
    config = {
        "created_at": utc_now(), "benchmark_id": BENCHMARK_ID,
        "models": MODELS_TO_RUN, "model_specs": MODEL_SPECS,
        "suite_hash": SUITE_HASH, "run_settings": RUN_SETTINGS,
        "approved_aggregate_budget_usd": BUDGET_USD,
    }
    if config_path.exists():
        prior = json.loads(config_path.read_text())
        for key in ("benchmark_id", "models", "model_specs", "suite_hash", "run_settings"):
            if prior.get(key) != config.get(key):
                raise RuntimeError(f"Resume configuration mismatch: {key}")
        if BUDGET_USD > float(prior["approved_aggregate_budget_usd"]):
            raise RuntimeError("A resume cannot raise the original budget; use a new output directory")
    else:
        write_json(config_path, config)
    existing = load_existing()
    if any(row.get("cost_usd") is None for row in existing.values()):
        raise RuntimeError("Cannot resume safely: a completed call is missing reported cost")
    return get_api_key(), existing, sum(float(row["cost_usd"]) for row in existing.values())

def save_call(*, model, call_id, payload, expected, metadata, existing, api_key, spent):
    key = (model, call_id)
    if key in existing:
        return existing[key], spent, None
    reservation = reserve_cost(payload)
    if spent + reservation > BUDGET_USD + 1e-12:
        return None, spent, f"STOP budget: spent ${spent:.6f}; next call reserves ${reservation:.6f}; cap ${BUDGET_USD:.6f}"
    call_dir = RERUN_DIR / "models" / model.replace("/", "--") / "calls" / call_id
    write_json(call_dir / "request.json", payload)
    status, response, latency = send_openrouter(payload, api_key)
    write_json(call_dir / "raw-response.json", response)
    append_jsonl(RERUN_DIR / "raw-responses.jsonl", {
        "model": model, "call_id": call_id, "http_status": status, "response": response,
    })
    if status < 200 or status >= 300 or "error" in response:
        write_json(call_dir / "error.json", {
            "model": model, "call_id": call_id, "http_status": status,
            "latency_seconds": latency, "response": response, "recorded_at": utc_now(),
        })
        return None, spent, f"STOP API error on {model}/{call_id}: HTTP {status}; rerun to resume"
    raw = content_of(response)
    parsed, strict = parse_command(raw)
    usage = usage_of(response)
    result = {
        "model": model, "returned_model": response.get("model"),
        "provider": response.get("provider"), "call_id": call_id,
        "suite_hash": SUITE_HASH, "expected": expected, "parsed": parsed,
        "strictly_formatted": strict, "correct": parsed == expected,
        "raw_content": raw, "latency_seconds": latency, "recorded_at": utc_now(),
        **metadata, **usage,
    }
    write_json(call_dir / "result.json", result)
    existing[key] = result
    if usage["cost_usd"] is None:
        return result, spent, "STOP accounting: response has no reported cost; rerun to resume"
    spent += float(usage["cost_usd"])
    return result, spent, None


In [ ]:
SUITE_HASH = stable_hash({"benchmark": BENCHMARK_ID, "jobs": [(j["id"], j["answer"], hashlib.sha256(j["image_bytes"]).hexdigest()) for j in JOBS], "models": MODELS_TO_RUN, "settings": RUN_SETTINGS})
def run_full_suite():
    api_key, existing, spent = prepare_run()
    stop = None
    total = len(MODELS_TO_RUN) * len(JOBS)
    for order, model in enumerate(MODELS_TO_RUN):
        for job_index, job in enumerate(JOBS):
            call_number = order * len(JOBS) + job_index + 1
            call_id = job["id"]
            if (model, call_id) not in existing:
                print(f"run {call_number:03d}/{total} {model}/{call_id}")
            payload = build_payload(model, job)
            _, spent, stop = save_call(
                model=model, call_id=call_id, payload=payload, expected=job["answer"],
                metadata=job_metadata(job), existing=existing, api_key=api_key, spent=spent,
            )
            if stop:
                print(stop)
                break
        if stop:
            break
    summary = {
        "updated_at": utc_now(), "completed_calls": len(existing), "expected_calls": total,
        "reported_cost_usd": sum(float(r["cost_usd"]) for r in existing.values() if r.get("cost_usd") is not None),
        "correct_calls": sum(r.get("correct") is True for r in existing.values()),
        "missing_cost_calls": sum(r.get("cost_usd") is None for r in existing.values()),
    }
    write_json(RERUN_DIR / "summary.json", summary)
    return summary

print(f"Exact tasks: {len(JOBS)}; selected models: {len(MODELS_TO_RUN)}; planned calls: {len(JOBS) * len(MODELS_TO_RUN)}")
largest_reservation = max(reserve_cost(build_payload(model, job)) for model in MODELS_TO_RUN for job in JOBS)
print(f"Largest conservative single-call reservation: ${largest_reservation:.4f}")
if RUN_PAID_EVAL:
    print(json.dumps(run_full_suite(), indent=2))
else:
    print("DRY RUN ONLY — set RUN_PAID_EVAL = True to send requests")


## Analyze your rerun


In [ ]:
import pandas as pd

rerun_rows = [json.loads(path.read_text()) for path in result_files()]
if not rerun_rows:
    print("No rerun results yet.")
else:
    rerun = pd.DataFrame(rerun_rows)
    display(rerun.groupby("model").agg(calls=("call_id", "size"), correct=("correct", "sum"), cost_usd=("cost_usd", "sum")))
